# Trabajo Práctico 2 - Grupo 02

### Modelo Bayes Naive

Integrantes:

*   Bermudez, Agustin
*   Calderón, Tiago
*   Gonzalez Pautaso, Mateo
*   Moreyra, Santiago
*   Nieves, Maylen

El objetivo es establecer un baseline con Multinomial Bayes Naive sobre dos representaciones de texto, Bag of Words y TF-IDF, con búsqueda de hiperparámetros.

NB es el modelo canónico para clasificación de texctos, es rápido de entrenar y transparente, ya que los pesos son log-probabilidades por palabra y clase.

Cualquier modelo mas sofisticado debe superar el F1-macro de NB, si no lo hace hay un bug.

## Importación e instalación de dependencias


In [ ]:
!pip install -q spacy
!python -m spacy download es_core_news_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 91.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

In [ ]:
from common.preprocessing import clean_classical
from common.data_utils import get_split, make_tfidf, make_bow, SEED
from common.evaluation import evaluate
from common.io_utils import save_model, load_model, make_submission

np.random.seed(SEED)

## Carga de datos y preprocesamiento

In [ ]:
train = pd.read_csv("/content/train.csv")
test  = pd.read_csv("/content/test.csv")
X_train_raw, X_val_raw, y_train, y_val = get_split(train)

In [ ]:
print("Preprocesando")
X_train = np.array([clean_classical(t) for t in X_train_raw])
X_val   = np.array([clean_classical(t) for t in X_val_raw])
X_test  = np.array([clean_classical(t) for t in test['text'].values])
print(f"Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

Preprocesando
Train: 40,800 | Val: 10,200 | Test: 8,500


## Entrega N3: Bayes Naive + TF-IDF con GridSearch


Se utiliza `GridSearchCV` para encontrar el mejor valor de `alpha`
(suavizado de Laplace) para Naive Bayes con representación TF-IDF.

`alpha` controla qué tan seguros está el modelo sobre palabras que
aparecen poco en el entrenamiento. Un alpha muy bajo puede hacer que
el modelo sea demasiado sensible a palabras raras; uno muy alto suaviza
demasiado y pierde discriminación.

GridSearch prueba 8 valores de alpha con validación cruzada de 5 folds
y elige el que maximiza el F1-macro promedio.

In [ ]:
pipe = Pipeline([
        ("tfidf", make_tfidf()),
        ("nb",    MultinomialNB())])

param_grid_tfidf_v1 = {
    "tfidf__ngram_range": [(1, 1), (1, 2)], # Rango de ngramas
    "tfidf__min_df": [3, 5, 10], # Cantidad minima de documentos donde debe aparecer la palabra
    "tfidf__sublinear_tf": [True, False], # True: 1 + log(tf)
    "nb__alpha": [0.1, 0.2, 0.3, 0.5], # Suavizado de Laplace
    "nb__fit_prior": [True, False] # False: priors uniformes
}

gs_tfidf_v1 = GridSearchCV(
    pipe,
    param_grid_tfidf_v1,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1
)

print("Buscando mejores hiperparámetros v1...")
gs_tfidf_v1.fit(X_train, y_train)

print(f"\nMejores hiperparámetros:: {gs_tfidf_v1.best_params_}")
print(f"Mejor F1-macro en CV: {gs_tfidf_v1.best_score_:.4f}")

Buscando mejores hiperparámetros v1...
Fitting 5 folds for each of 96 candidates, totalling 480 fits

Mejores hiperparámetros:: {'nb__alpha': 0.5, 'nb__fit_prior': False, 'tfidf__min_df': 10, 'tfidf__ngram_range': (1, 2), 'tfidf__sublinear_tf': False}
Mejor F1-macro en CV: 0.6497


In [ ]:
best_tfidf_v1 = gs_tfidf_v1.best_estimator_
y_pred_tfidf_v1 = best_tfidf_v1.predict(X_val)

evaluate(
    "nb_tfidf_gridsearch_v1",
    y_val,
    y_pred_tfidf_v1,
    hyperparams={
        **gs_tfidf_v1.best_params_,
        "vectorizer": "TF-IDF",
        "cv_folds": 5
    }
)


=== nb_tfidf_gridsearch_v1 ===
Hiperparámetros: {'nb__alpha': 0.5, 'nb__fit_prior': False, 'tfidf__min_df': 10, 'tfidf__ngram_range': (1, 2), 'tfidf__sublinear_tf': False, 'vectorizer': 'TF-IDF', 'cv_folds': 5}

F1-macro:  0.6501
Precision: 0.6573
Recall:    0.6532
Accuracy:  0.6812

              precision    recall  f1-score   support

    negativa     0.7708    0.7091    0.7387      4080
      neutra     0.3706    0.5132    0.4304      2040
    positiva     0.8305    0.7373    0.7811      4080

    accuracy                         0.6812     10200
   macro avg     0.6573    0.6532    0.6501     10200
weighted avg     0.7147    0.6812    0.6940     10200

Matriz de confusión (filas=real, cols=predicho):
          negativa  neutra  positiva
negativa      2893    1002       185
neutra         564    1047       429
positiva       296     776      3008


In [ ]:
best_tfidf_v1.fit(
    np.concatenate([X_train, X_val]),
    np.concatenate([y_train, y_val])
)

save_model(best_tfidf_v1, "nb_tfidf_gridsearch_power_v1")
make_submission(
    best_tfidf_v1,
    pd.DataFrame({"id": test["id"], "text": X_test}),
    "submission_nb_tfidf_gridsearch_power_v1.csv"
);

Modelo guardado → models/nb_tfidf_gridsearch_power_v1.joblib
Predicción guardada → submissions/submission_nb_tfidf_gridsearch_power_v1.csv  (8500 predicciones)
Distribución: clase 0: 36.6%, clase 1: 27.3%, clase 2: 36.1%
